# # Load training data

In [1]:
from src.demo import prepare
conn_name = prepare()

from nubison_model import data

db = data.connection(conn_name)
df = db.load("SELECT * FROM iris")
datasets = data.split(df, {"training": 0.8, "evaluation": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:12s} rows={len(sub):3d}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


training     rows=120

evaluation   rows= 30

# # Train (PyTorch Lightning)

In [2]:
# `train(datasets, target, *, ...)` parameters:
#   datasets      — dict from `data.split` (must include "training";
#                   "evaluation" enables `t.X_eval / t.y_eval` + auto-scoring)
#   target        — label column name(s); single string or list of strings
#   model_type    — free-form string tagged on the run as `model_type`
#                   (surfaced in the nubison UI). "classifier" and
#                   "regressor" additionally make `t.fit()` log
#                   `evaluation_accuracy` / `evaluation_r2`. None skips
#                   the tag.
#   weights_path  — where `t.save(model)` writes the pickle.
#                   Default "src/weights.pkl" matches `register(artifact_dirs="src")`.
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl
from nubison_model import train


class IrisLightning(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.fc(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log("loss", loss, prog_bar=False)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.01)


with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    X = torch.tensor(t.X.values, dtype=torch.float32)
    y = torch.tensor(t.y.values.ravel(), dtype=torch.long)
    loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)

    model = IrisLightning()
    trainer = pl.Trainer(
        max_epochs=30,
        accelerator="cpu",
        enable_progress_bar=False,
        logger=False,
        enable_checkpointing=False,
    )
    trainer.fit(model, loader)

    t.save(model)

print(f"run_id: {t.run_id}")


2026/05/15 19:42:46 INFO mlflow.tracking.fluent: Experiment with name 'ListTarget_train_lightning_1778841757' does not exist. Creating a new experiment.


2026/05/15 19:42:46 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


2026/05/15 19:42:46 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


2026/05/15 19:42:46 INFO mlflow.tracking.fluent: Autologging successfully enabled for pytorch_lightning.


2026/05/15 19:42:46 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


...

/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run valuable-dolphin-226 at: http://127.0.0.1:5050/#/experiments/31/runs/3039dacd8ac84d869c23173858f2c628


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/31


run_id: 3039dacd8ac84d869c23173858f2c628